# Forest cover Predictor 25/26 - Project for Data Science and AI for Business
Authors: Andreea Patarlageanu, Harshita Goyal, Anirudh Sudhir, Benedikt Watzinger

In [ ]:
# We will put all imports here
import pandas as pd
import numpy as np
import os
import ydata_profiling

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

## 1. Exploratory Data Analysis

We start with an exploratoy data analysis in order to better understand the data and later to format it efficiently for good results.

### Loading data, checking features

#### Load data

In [ ]:
file_path = "data/train.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Loaded data in df.")
else:
    print("File does not exist.")

In [ ]:
df.shape

In [ ]:
df.head(5)

We see that we have 15120 and more than 50 features. The dimension is quite large, so we need to inspect these features and later 'eliminate' some of them for dimension reduction.

#### Test file

Let's see if the test file is similar to the train one.

In [ ]:
test_file_path = "data/test-full.csv"
if os.path.exists(test_file_path):
    df_test = pd.read_csv(test_file_path)
    print("Loaded test data in df_test.")

In [ ]:
df_test.shape

In [ ]:
df_test.columns

Let's check easily if indeed both $df$ and $df_{test}$ have the same columns:

In [ ]:
df_cols = set(df.columns) - {"Cover_Type"}
df_test_cols = set(df_test.columns)

print("Same columns?", df_cols == df_test_cols)

#### Inspecting the submission file structure

Let's see hwo the submission file looks like, so we know what is expected:

In [ ]:
target = pd.read_csv("data/full_submission.csv")
print(f"Length of submission example: {len(target)} rows.")

In [ ]:
target.head(5)

Now we know that it is expected from us to "de;iver" a file with 2 columns, the Id and the predicted Cover_Type.

## 2. Data inspection

### Data types

Let's inspect the content of the columns and adjust data types if necessary.

In [ ]:
print(df.columns)
print(df.dtypes)

Great! All of them are integer type.

### Missing values

In [ ]:
df.isnull().sum()

There are no missing values! This is great, no modification needed.

## 3. Feature analysis

### Distribution of features

Let's inspect the distribution of the features, except the target.

In [ ]:
numerical_features = df.select_dtypes(include=["int64"]).columns.tolist()
numerical_features.remove("Cover_Type")
numerical_features.remove("Id")

In [ ]:
binary_features = []
continuous_features = []
trivial_featues = []

print("TRAIN SET ANALYSIS")
print("="*100)

for feature in numerical_features:
    data = df[feature]

    # We are interested in some "basic" statistics
    n_unique = data.nunique()
    min_val = data.min()
    max_val = data.max()

    if n_unique == 1:
        trivial_featues.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range {data.unique()}"
        )

    # How many unique values? Important to see if binary/categorical
    elif n_unique == 2:
        binary_features.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range {data.sort_values().unique()}"
        )

    else:
        continuous_features.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range [{min_val:4.0f} : {max_val:4.0f}]"
        )

In [ ]:
print("TEST SET ANALYSIS")
print("="*100)

for feature in numerical_features:
    data = df_test[feature]

    # We are interested in some "basic" statistics
    n_unique = data.nunique()
    min_val = data.min()
    max_val = data.max()

    if n_unique == 1:
        # trivial_featues.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range {data.unique()}"
        )

    # How many unique values? Important to see if binary/categorical
    elif n_unique == 2:
        # binary_features.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range {data.sort_values().unique()}"
        )

    else:
        # continuous_features.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range [{min_val:4.0f} : {max_val:4.0f}]"
        )

In [ ]:
binary_features

In [ ]:
trivial_featues

We observe that:

- The 4 types of Wilderness Areas are binary.
- All soil types are binary.
- Soil Type 15 takes only the 0 value, so brings no information. We can remove it.

In [ ]:
# df = df.drop(trivial_featues, axis=1)

#### Continuous features

Let's better visualize the continuous features:

In [ ]:
sns.set_style("whitegrid")

nr_features = len(continuous_features)
n_cols = 3
n_rows = (nr_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):

    data_to_plot = df[feature].dropna()

    unique_count = data_to_plot.nunique()

    if unique_count < 30:
        bins = unique_count
        kde = False
    else:
        bins = min(int(np.ceil(np.log2(len(data_to_plot)) + 1)), 50)
        kde = True

    sns.histplot(data=data_to_plot, kde=kde, ax=axes[idx], bins=bins, color="steelblue")

    mean_val = data_to_plot.mean()
    median_val = data_to_plot.median()

    axes[idx].axvline(
        mean_val,
        color="red",
        linestyle="--",
        linewidth=2,
        alpha=0.7,
        label=f"Mean: {mean_val:.1f}",
    )
    axes[idx].axvline(
        median_val,
        color="orange",
        linestyle="--",
        linewidth=2,
        alpha=0.7,
        label=f"Median: {median_val:.1f}",
    )

    axes[idx].set_title(f"{feature}", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel(feature.replace("_", " "), fontsize=9)
    axes[idx].set_ylabel("Frequency", fontsize=9)
    axes[idx].legend(fontsize=7, loc="best")
    axes[idx].grid(axis="y", alpha=0.3)

for idx in range(nr_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle("Continuous Feature Distributions", fontsize=16, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig("output/plots/continuous_feature_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

We observe that the "Horizontal_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways" and "Horizontal_Distance_To_Fire_Points" are severely right-skewed, having extreme outliers. 

One option would be to **Log transform** the right skewed features, to reduce overfitting on outliers!

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("whitegrid")

nr_features = len(continuous_features)
n_cols = 3
n_rows = (nr_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):
    train_data = df[feature].dropna()
    test_data = df_test[feature].dropna()

    # Decide bins & KDE
    unique_count = max(train_data.nunique(), test_data.nunique())
    if unique_count < 30:
        bins = unique_count
        kde = False
    else:
        bins = min(int(np.ceil(np.log2(len(train_data)) + 1)), 50)
        kde = True

    # Plot normalized histograms
    sns.histplot(train_data, bins=bins, kde=kde, ax=axes[idx],
                 color="steelblue", label="Train", alpha=0.5, stat="density")
    sns.histplot(test_data, bins=bins, kde=kde, ax=axes[idx],
                 color="orange", label="Test", alpha=0.4, stat="density")

    # Optional: Mean & median lines (uncomment if needed)
    # axes[idx].axvline(train_data.mean(), color="blue", linestyle="--", linewidth=1.5, label=f"Train Mean")
    # axes[idx].axvline(train_data.median(), color="darkblue", linestyle=":", linewidth=1.5, label=f"Train Median")
    # axes[idx].axvline(test_data.mean(), color="red", linestyle="--", linewidth=1.5, label=f"Test Mean")
    # axes[idx].axvline(test_data.median(), color="darkred", linestyle=":", linewidth=1.5, label=f"Test Median")

    # Titles and labels
    axes[idx].set_title(f"{feature}", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel(feature.replace("_", " "), fontsize=9)
    axes[idx].set_ylabel("Density", fontsize=9)
    axes[idx].legend(fontsize=7, loc="best")  # ← Legend added
    axes[idx].grid(axis="y", alpha=0.3)

# Remove empty subplots
for idx in range(nr_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle("Train vs Test Feature Distributions", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("output/plots/train_test_feature_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

#### Wilderness Areas

Let's visualize the Wilderness Areas better. From the description of the data given on Kaggle, we know the following names:

In [ ]:
wilderness_features = [
    "Wilderness_Area1",
    "Wilderness_Area2",
    "Wilderness_Area3",
    "Wilderness_Area4",
]

wilderness_names = {
    "Wilderness_Area1": "WA1 - Rawah",
    "Wilderness_Area2": "WA2 - Neota",
    "Wilderness_Area3": "WA3 - Comanche Peak",
    "Wilderness_Area4": "WA4 - Cache la Poudre",
}

In [ ]:
import matplotlib.pyplot as plt

# Data
wilderness_counts = [df[f].sum() for f in wilderness_features]
wilderness_labels = [wilderness_names[f] for f in wilderness_features]
colors = sns.color_palette("viridis", 4)

# Plot horizontal bar chart
plt.figure(figsize=(10, 5))
bars = plt.barh(wilderness_labels, wilderness_counts, color=colors, edgecolor="black")

# Invert y-axis to flip sequence
plt.gca().invert_yaxis()

# Add labels with percentages
for bar, count in zip(bars, wilderness_counts):
    width = bar.get_width()
    pct = count / len(df) * 100
    x_pos = width + 100 if width < 1000 else width / 2
    ha = "left" if width < 1000 else "center"
    va = "center"

    plt.text(
        x_pos,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,.0f}\n({pct:.1f}%)",
        ha=ha,
        va=va,
        fontsize=10,
        fontweight="bold",
    )

# Reference line for perfect balance (25% each)
perfect_balance = len(df) / 4
plt.axvline(perfect_balance, color="red", linestyle="--", linewidth=2, alpha=0.5, label="Perfect Balance (25%)")
plt.legend()

plt.title("Wilderness Area Distribution in Train Set)")
plt.xlabel("Count")
plt.ylabel("Wilderness Area")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data
wilderness_counts = [df_test[f].sum() for f in wilderness_features]
wilderness_labels = [wilderness_names[f] for f in wilderness_features]
colors = sns.color_palette("viridis", 4)

# Plot horizontal bar chart
plt.figure(figsize=(10, 5))
bars = plt.barh(wilderness_labels, wilderness_counts, color=colors, edgecolor="black")

# Invert y-axis to flip sequence
plt.gca().invert_yaxis()

# Add labels with percentages
for bar, count in zip(bars, wilderness_counts):
    width = bar.get_width()
    pct = count / len(df_test) * 100
    x_pos = width + 100 if width < 1000 else width / 2
    ha = "left" if width < 1000 else "center"
    va = "center"

    plt.text(
        x_pos,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,.0f}\n({pct:.1f}%)",
        ha=ha,
        va=va,
        fontsize=10,
        fontweight="bold",
    )

# Reference line for perfect balance (25% each)
perfect_balance = len(df_test) / 4
plt.axvline(perfect_balance, color="red", linestyle="--", linewidth=2, alpha=0.5, label="Perfect Balance (25%)")
plt.legend()

plt.title("Wilderness Area Distribution in Test Set")
plt.xlabel("Count")
plt.ylabel("Wilderness Area")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Data
wilderness_labels = [wilderness_names[f] for f in wilderness_features]

# Compute counts and normalize
counts_train = np.array([df[f].sum() for f in wilderness_features])
counts_test  = np.array([df_test[f].sum() for f in wilderness_features])

counts_train_pct = counts_train / counts_train.sum() * 100
counts_test_pct  = counts_test / counts_test.sum() * 100

# Colors
colors = sns.color_palette("viridis", 4)

# Reverse labels and data for inverted category sequence
wilderness_labels_rev = wilderness_labels[::-1]
counts_train_rev = counts_train_pct[::-1]
counts_test_rev = counts_test_pct[::-1]

# Create subplots: 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Train plot
axes[0].barh(wilderness_labels_rev, counts_train_rev, color=colors, edgecolor="black")
axes[0].set_title("Train Set")
axes[0].set_xlabel("Percentage")
axes[0].set_xlim(0, max(max(counts_train_pct), max(counts_test_pct)) * 1.1)

# Add percentage labels
for i, pct in enumerate(counts_train_rev):
    axes[0].text(pct + 0.5, i, f"{pct:.1f}%", va='center', ha='left', fontsize=10, fontweight='bold')

# Test plot
axes[1].barh(wilderness_labels_rev, counts_test_rev, color=colors, edgecolor="black")
axes[1].set_title("Test Set")
axes[1].set_xlabel("Percentage")
axes[1].set_xlim(0, max(max(counts_train_pct), max(counts_test_pct)) * 1.1)

# Add percentage labels
for i, pct in enumerate(counts_test_rev):
    axes[1].text(pct + 0.5, i, f"{pct:.1f}%", va='center', ha='left', fontsize=10, fontweight='bold')

plt.suptitle("Wilderness Area Distribution: Train vs Test", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("output/plots/wilderness_area_distribution_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

We observe that the Wilderness Area is highly imbalanced, especially for the Neota type, which represents only 3.8%. We added a dotted line which hows the level for a perfect balance. 

However, since Wilderness Area is a feature, we should keep in mind this imbalance when trying to predict the Cover type, but for now we leave it as it is.

### Target variable distribution

Let's seethe distribution of the target variable:

In [ ]:
# Simple check: Are all 7 classes balanced?
df["Cover_Type"].value_counts().sort_index()

In [ ]:
distribution_cover_type = df["Cover_Type"].value_counts().sort_index()
sns.barplot(
    x=distribution_cover_type.index,
    y=distribution_cover_type.values,
    hue=distribution_cover_type.index,
    palette=sns.color_palette("twilight", 7),
    legend=False,
)
plt.xlabel("Cover Type")
plt.ylabel("Count")
plt.title("Cover Type Distribution (Overall for Train Set)")
plt.savefig("output/plots/cover_type_distribution_global.png", dpi=1000, bbox_inches="tight")
plt.show()

In [ ]:
df["Wilderness_Area"] = df["Wilderness_Area1"] * 1 + df["Wilderness_Area2"] * 2 + df["Wilderness_Area3"] * 3 + df["Wilderness_Area4"] * 4

In [ ]:
# Create pivot table of counts
distribution_cover_type_by_wtype = (
    df.pivot_table(
        index="Wilderness_Area",
        columns="Cover_Type",
        values="Id",
        aggfunc="count",
    )
    # Normalize within each Wilderness Area (row-wise)
    .pipe(lambda x: x.div(x.sum(axis=1), axis=0))
    .reset_index()
    .melt(
        id_vars="Wilderness_Area",
        var_name="Cover_Type",
        value_name="Proportion"
    )
)

# Plot normalized (proportion) bars
g = sns.catplot(
    data=distribution_cover_type_by_wtype,
    x="Cover_Type",
    y="Proportion",
    hue="Cover_Type",
    palette=sns.color_palette("twilight", 7),
    col="Wilderness_Area",
    kind="bar",
    col_wrap=2,
    height=4,
    aspect=1,
    legend=False
)

g.figure.suptitle("Cover Type Distribution by Wilderness Area", y=1.02)
g.set_ylabels("Proportion")  # Update y-axis label to reflect normalization

plt.savefig("output/plots/cover_type_distribution_by_wtype.png", dpi=1000, bbox_inches="tight")
plt.show()

### Feature distribution by Cover Type

In [ ]:
continuous_features

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 10))
axes = axes.flatten()

# features = ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology"]
# colors = ["steelblue", "coral", "lightgreen", "plum"]

for idx, feature in enumerate(continuous_features):
    sns.boxplot(data=df, x="Cover_Type", y=feature, ax=axes[idx])
    axes[idx].set_title(feature)

plt.tight_layout()
plt.show()

**Comment:**

- Elevation might be a strong predictor because of its non-linear relationship for all the Cover Types.
- Aspect by itself doesn't differentiate much the Cover types, but might be useful in interactions.
- Slope can be a moderate predictor, making some differences between Types 1, 2, 7 and 3, 4, 5, 6.
- Horizontal Distance To Hydrology might be an excellent predictor, clearly separating Type 4 of the rest.

### Correlation with the target

In [ ]:
correlations = df.corr()["Cover_Type"].drop("Cover_Type").sort_values(ascending=False)
print(correlations)

In [ ]:
# Compute correlation matrix
corr_matrix = df[continuous_features].corr()

# Create mask for diagonal entries
mask = np.eye(len(corr_matrix), dtype=bool)  # True on diagonal

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    mask=mask,  # hide diagonal
    square=True,
    linewidths=.5,
    cbar_kws={"shrink": .8}
)
plt.title("Feature-to-Feature Correlation Matrix")
plt.savefig("output/plots/feature_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

The correlations coefficients range from -1 to 1 and they have the following meaning:

1. if it is close to -1, then it has perfect negative correlation
2. close to 0 means non-linear relationship 
3. close to 1 means perfect positive correlation

In [ ]:
# check that each row has exactly one soil type
df[[col for col in df.columns if col.startswith("Soil_Type")]].sum(axis=1).value_counts()

In [ ]:
df.columns